In [5]:
!wget https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

--2025-12-09 14:44:30--  https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg
Resolving habrastorage.org (habrastorage.org)... 95.47.173.34, 95.47.173.35, 2a14:b680:0:56::34, ...
Connecting to habrastorage.org (habrastorage.org)|95.47.173.34|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 398272 (389K) [image/jpeg]
Saving to: ‘yf_dokzqy3vcritme8ggnzqlvwa.jpeg’

yf_dokzqy3vcritme8g 100%[===================>] 388.94K   748KB/s    in 0.5s    

2025-12-09 14:44:31 (748 KB/s) - ‘yf_dokzqy3vcritme8ggnzqlvwa.jpeg’ saved [398272/398272]



In [ ]:
!pip install onnx

In [ ]:
!pip install onnxruntime

In [ ]:
import onnxruntime as ort

In [4]:
import onnx

model = onnx.load('/content/hair_classifier_v1.onnx')
model

ModelProto(ir_version=10, opset_import={'': 20}, producer_name='pytorch', producer_version='2.9.0+cu126', graph=GraphProto('main_graph', input=<1 inputs>, output=<1 outputs>, initializer=<7 initializers>, node=<9 nodes>, value_info=<15 value_info>))

In [8]:
import numpy as np
from io import BytesIO
from urllib import request
from PIL import Image
import onnxruntime as ort

# Load the model with the correct path
model_path = 'hair_classifier_v1.onnx'
session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

def preprocess_input(img):
    # Convert to numpy array, explicitly ensuring float32
    x = np.array(img, dtype=np.float32)

    # Normalize to [0, 1]
    x = x / 255.0

    # ImageNet standardization, explicitly ensuring float32 for mean and std
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    x = (x - mean) / std

    # Transpose to channel-first format (C, H, W) and add batch dimension
    x = x.transpose(2, 0, 1)
    x = np.expand_dims(x, axis=0)

    return x

def predict(url):
    # Download and prepare image
    img = download_image(url)
    img = prepare_image(img, target_size=(200, 200))

    # Preprocess
    X = preprocess_input(img)

    # Get input name
    input_name = session.get_inputs()[0].name

    # Run inference
    outputs = session.run(None, {input_name: X})

    return float(outputs[0][0])

# Image URL for prediction
image_url = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"

# Perform prediction
prediction_result = predict(image_url)
print(f"The model prediction for the image is: {prediction_result}")

The model prediction for the image is: 0.09156641364097595


/tmp/ipython-input-1444616883.py:57: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(outputs[0][0])


In [9]:
# Re-using the functions defined previously in the notebook
# image_url is also available from previous execution

# Download and prepare image
img = download_image(image_url)
img = prepare_image(img, target_size=(200, 200))

# Preprocess
X = preprocess_input(img)

# Get the R channel value of the first pixel (index [0, 0, 0, 0])
# X has shape (Batch, Channel, Height, Width)
first_pixel_r_channel = X[0, 0, 0, 0]

print(f"The R channel value of the first pixel after preprocessing is: {first_pixel_r_channel}")

The R channel value of the first pixel after preprocessing is: -1.0732940435409546


In [ ]:
# Get input names
for input in model.graph.input:
    print(f"Input: {input.name}")

# Get output names
for output in model.graph.output:
    print(f"Output: {output.name}")

In [ ]:
import onnx

model = onnx.load('hair_classifier_v1.onnx')

# Get input names
for input in model.graph.input:
    print(f"Input: {input.name}")

# Get output names
for output in model.graph.output:
    print(f"Output: {output.name}")

Input: input
Output: output


In [ ]:
!pip install pillow

In [ ]:
!wget https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

In [ ]:
import numpy as np
from io import BytesIO
from urllib import request
from PIL import Image

# Load the model
model_path = 'hair_classifier_empty.onnx'
session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

def preprocess_input(img):
    # Convert to numpy array
    x = np.array(img, dtype=np.float32)

    # Normalize to [0, 1]
    x = x / 255.0

    # ImageNet standardization
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    x = (x - mean) / std

    # Transpose to channel-first format (C, H, W) and add batch dimension
    x = x.transpose(2, 0, 1)
    x = np.expand_dims(x, axis=0)

    return x

def predict(url):
    # Download and prepare image
    img = download_image(url)
    img = prepare_image(img, target_size=(200, 200))

    # Preprocess
    X = preprocess_input(img)

    # Get input name
    input_name = session.get_inputs()[0].name

    # Run inference
    outputs = session.run(None, {input_name: X})

    return float(outputs[0][0])

def lambda_handler(event, context):
    url = event['url']
    result = predict(url)

    return {
        'statusCode': 200,
        'body': {'prediction': result}
    }